# 86BB Bio-Score Viability Analysis

Can we replace preset interpolation with a continuous 12D biological feature vector?

This notebook tests:
1. Do the 16 cluster centroids produce meaningfully distinct bio-score vectors?
2. What do random points in gut space look like as bio-scores?
3. How do bio-scores evolve during a simulated food-selection session?
4. Is the 12D space well-structured for regression-based A/V mapping?

In [1]:
import zipfile
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform, cosine
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

PLOTLY_DARK = 'plotly_dark'
OUT_DIR = Path('../outputs')
SUPP_DIR = Path('../data/predict1_paper/supplementary')
TABLE5_FILE = SUPP_DIR / '41591_2020_1183_MOESM6_ESM.xlsx'
TABLE9_FILE = SUPP_DIR / '41591_2020_1183_MOESM10_ESM.xlsx'

important_species = pd.read_csv(OUT_DIR / 'important_species_list.csv')['species'].tolist()
corr_matrix = pd.read_csv(OUT_DIR / 'predict1_food_species_correlations.csv', index_col=0)
health_dir_df = pd.read_csv(OUT_DIR / 'health_direction_vector.csv', index_col=0)
food_profiles_df = pd.read_csv(OUT_DIR / 'food_species_profiles.csv', index_col=0)
cluster_profiles = pd.read_csv(OUT_DIR / 'cluster_profiles.csv', index_col=0)
cluster_centroids = pd.read_csv(OUT_DIR / 'cluster_centroids.csv', index_col=0)
fasting_matrix = pd.read_csv(OUT_DIR / 'fasting_species_correlations.csv', index_col=0)
postprandial_matrix = pd.read_csv(OUT_DIR / 'postprandial_species_correlations.csv', index_col=0)
weight_matrix = pd.read_csv(OUT_DIR / 'food_category_weights.csv', index_col=0)

FOOD_GROUPS = list(corr_matrix.columns)
corr_np = corr_matrix.loc[important_species].values.T
health_direction = health_dir_df['health_weight'].reindex(important_species).fillna(0).values

NS = {'main': 'http://purl.oclc.org/ooxml/spreadsheetml/main'}
def load_strict_xlsx(filepath, sheet_index=0):
    with zipfile.ZipFile(filepath) as z:
        ss_xml = z.read('xl/sharedStrings.xml').decode('utf-8')
        ss_root = ET.fromstring(ss_xml)
        strings = []
        for si in ss_root.findall('.//main:si', NS):
            parts = [t.text for t in si.findall('.//main:t', NS) if t.text]
            strings.append(''.join(parts))
        sheet_file = f'xl/worksheets/sheet{sheet_index + 1}.xml'
        xml = z.read(sheet_file).decode('utf-8')
        root = ET.fromstring(xml)
        rows = []
        for row_el in root.findall('.//main:row', NS):
            cells = {}
            for cell in row_el.findall('main:c', NS):
                ref = cell.attrib.get('r', '')
                col = ''.join(c for c in ref if c.isalpha())
                val_el = cell.find('main:v', NS)
                if val_el is None: continue
                cell_type = cell.attrib.get('t', '')
                if cell_type == 's': cells[col] = strings[int(val_el.text)]
                else:
                    try: cells[col] = float(val_el.text)
                    except ValueError: cells[col] = val_el.text
            if cells: rows.append(cells)
        return rows

def rows_to_df(rows, skip_header=3):
    data = rows[skip_header:]
    df = pd.DataFrame(data)
    df.columns = ['variable', 'species', 'spearman_r', 'pvalue', 'qvalue']
    df['spearman_r'] = pd.to_numeric(df['spearman_r'], errors='coerce')
    df['pvalue'] = pd.to_numeric(df['pvalue'], errors='coerce')
    df['qvalue'] = pd.to_numeric(df['qvalue'], errors='coerce')
    return df.dropna(subset=['spearman_r'])

pattern_df_raw = rows_to_df(load_strict_xlsx(TABLE5_FILE, 2))
pattern_matrix = pattern_df_raw.pivot(index='species', columns='variable', values='spearman_r')
pattern_135 = pattern_matrix.reindex(important_species).fillna(0)
fasting_135 = fasting_matrix.reindex(important_species).fillna(0)
postprandial_135 = postprandial_matrix.reindex(important_species).fillna(0)

print(f'Loaded: {len(important_species)} species, {fasting_135.shape[1]} fasting markers, {postprandial_135.shape[1]} pp markers')

Loaded: 135 species, 75 fasting markers, 169 pp markers


---
## 1. Define the Bio-Score Function

12 continuous biological scores computed from any 135D gut profile.

In [2]:
NEUROACTIVE_PATHWAYS = {
    'butyrate_SCFA': [
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis', 'Roseburia_intestinalis',
        'Roseburia_faecis', 'Roseburia_sp_CAG_182', 'Roseburia_sp_CAG_309',
        'Roseburia_sp_CAG_471', 'Eubacterium_hallii', 'Eubacterium_eligens',
        'Coprococcus_eutactus', 'Coprococcus_comes', 'Coprococcus_catus',
        'Anaerostipes_hadrus', 'Agathobaculum_butyriciproducens',
        'Lachnospira_pectinoschiza', 'Eubacterium_ramulus',
        'Intestinimonas_butyriciproducens',
    ],
    'GABA': [
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Bifidobacterium_catenulatum',
        'Bifidobacterium_pseudocatenulatum', 'Bifidobacterium_animalis',
        'Lactococcus_lactis', 'Bacteroides_fragilis',
    ],
    'tryptophan_serotonin': [
        'Streptococcus_thermophilus', 'Streptococcus_salivarius',
        'Streptococcus_parasanguinis', 'Eubacterium_hallii',
        'Faecalibacterium_prausnitzii', 'Roseburia_hominis',
        'Coprococcus_eutactus', 'Coprococcus_comes',
    ],
    'dopamine_catecholamine': ['Escherichia_coli'],
    'pro_inflammatory': [
        'Bilophila_wadsworthia', 'Ruminococcus_gnavus', 'Ruminococcus_torques',
        'Clostridium_bolteae', 'Clostridium_bolteae_CAG_59', 'Clostridium_symbiosum',
        'Hungatella_hathewayi', 'Escherichia_coli', 'Clostridium_innocuum',
        'Flavonifractor_plautii', 'Eggerthella_lenta',
    ],
    'gut_barrier': [
        'Akkermansia_muciniphila', 'Faecalibacterium_prausnitzii',
        'Bifidobacterium_longum', 'Bifidobacterium_adolescentis',
        'Bifidobacterium_bifidum', 'Roseburia_intestinalis',
    ],
}

for p in NEUROACTIVE_PATHWAYS:
    NEUROACTIVE_PATHWAYS[p] = [s for s in NEUROACTIVE_PATHWAYS[p] if s in important_species]

glyca_col = [c for c in fasting_135.columns if 'GlycA' in c]
lipid_adverse_cols = [c for c in fasting_135.columns if any(k in c for k in ['LDL', 'VLDL', 'Total_TG'])]
hpdi_col = 'hPDI' if 'hPDI' in pattern_135.columns else None

BIO_SCORE_NAMES = [
    'health', 'butyrate', 'GABA', 'serotonin', 'dopamine',
    'inflammation_species', 'gut_barrier', 'fasting_inflammation',
    'lipid_adverse', 'diet_hPDI', 'diet_uPDI', 'diversity_proxy'
]

def compute_bio_scores(gut_135, diet_history_groups=None):
    gut = pd.Series(gut_135, index=important_species) if not isinstance(gut_135, pd.Series) else gut_135

    scores = {}
    scores['health'] = float(gut.values @ health_direction)

    for pathway, species_list in NEUROACTIVE_PATHWAYS.items():
        if species_list:
            scores[pathway] = float(gut[species_list].mean())
        else:
            scores[pathway] = 0.0

    if glyca_col:
        vals = fasting_135[glyca_col].values.T @ gut.values
        scores['fasting_inflammation'] = float(np.mean(vals))
    else:
        scores['fasting_inflammation'] = 0.0

    if lipid_adverse_cols:
        vals = fasting_135[lipid_adverse_cols].values.T @ gut.values
        scores['lipid_adverse'] = float(np.mean(vals))
    else:
        scores['lipid_adverse'] = 0.0

    if hpdi_col:
        scores['diet_hPDI'] = float(pattern_135[hpdi_col].values @ gut.values)
    else:
        scores['diet_hPDI'] = 0.0

    updi_col = 'uPDI' if 'uPDI' in pattern_135.columns else None
    if updi_col:
        scores['diet_uPDI'] = float(pattern_135[updi_col].values @ gut.values)
    else:
        scores['diet_uPDI'] = 0.0

    if diet_history_groups and len(diet_history_groups) > 0:
        counts = np.zeros(len(FOOD_GROUPS))
        for fg in diet_history_groups:
            if fg in FOOD_GROUPS:
                counts[FOOD_GROUPS.index(fg)] += 1
        p = counts / counts.sum()
        p = p[p > 0]
        scores['diversity_proxy'] = float(-np.sum(p * np.log(p)) / np.log(len(FOOD_GROUPS)))
    else:
        scores['diversity_proxy'] = float(np.abs(gut.values).std())

    return scores

test_gut = cluster_centroids.iloc[0][important_species].values
test_scores = compute_bio_scores(test_gut)
print('Bio-score dimensions:', len(test_scores))
print('Score names:', list(test_scores.keys()))
print(f'\nTest (cluster 0): {test_scores}')

Bio-score dimensions: 12
Score names: ['health', 'butyrate_SCFA', 'GABA', 'tryptophan_serotonin', 'dopamine_catecholamine', 'pro_inflammatory', 'gut_barrier', 'fasting_inflammation', 'lipid_adverse', 'diet_hPDI', 'diet_uPDI', 'diversity_proxy']

Test (cluster 0): {'health': 1.616308742150089, 'butyrate_SCFA': 0.007824920609461406, 'GABA': 0.012613783704365238, 'tryptophan_serotonin': 0.03322412466401482, 'dopamine_catecholamine': -0.0449200507094073, 'pro_inflammatory': -0.03649968071406639, 'gut_barrier': 0.0041113056916998, 'fasting_inflammation': -0.23031444597001816, 'lipid_adverse': -0.1425352955968191, 'diet_hPDI': 0.147209329494995, 'diet_uPDI': -0.20988376190033867, 'diversity_proxy': 0.023746308432949094}


---
## 2. Cluster Centroids as Bio-Score Vectors

Do the 16 clusters produce meaningfully distinct 12D vectors?

In [3]:
K = len(cluster_centroids)
cluster_bio = pd.DataFrame(index=range(K))

for ci in range(K):
    gut = cluster_centroids.iloc[ci][important_species].values
    scores = compute_bio_scores(gut)
    for k, v in scores.items():
        cluster_bio.loc[ci, k] = v

cluster_bio.index.name = 'cluster'
print('Cluster bio-score matrix:')
print(cluster_bio.round(4).to_string())

Cluster bio-score matrix:
         health  butyrate_SCFA    GABA  tryptophan_serotonin  dopamine_catecholamine  pro_inflammatory  gut_barrier  fasting_inflammation  lipid_adverse  diet_hPDI  diet_uPDI  diversity_proxy
cluster                                                                                                                                                                                        
0        1.6163         0.0078  0.0126                0.0332                 -0.0449           -0.0365       0.0041               -0.2303        -0.1425     0.1472    -0.2099           0.0237
1        2.9688         0.0507 -0.0067                0.0288                 -0.0827           -0.0619      -0.0044               -0.3800        -0.2335     0.4614    -0.4426           0.0260
2       -1.3921        -0.0247 -0.0069               -0.0188                  0.0161            0.0240      -0.0001                0.1495         0.1029    -0.3123     0.2101           0.0186
3       -1.199

In [4]:
bio_normed = cluster_bio.apply(lambda x: (x - x.mean()) / x.std() if x.std() > 0 else 0)

fig = px.imshow(bio_normed.values,
    x=list(cluster_bio.columns), y=[f'C{i}' for i in range(K)],
    color_continuous_scale='RdBu_r', aspect='auto',
    title='16 Cluster Centroids as Bio-Score Vectors (z-scored)',
    template=PLOTLY_DARK, width=950, height=550)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [5]:
dist_matrix = squareform(pdist(cluster_bio.values, metric='cosine'))
print('Pairwise cosine distances between cluster bio-score vectors:')
print(f'  Min (excl 0): {dist_matrix[dist_matrix > 0].min():.4f}')
print(f'  Max: {dist_matrix.max():.4f}')
print(f'  Mean: {dist_matrix[np.triu_indices(K, k=1)].mean():.4f}')
print(f'  Median: {np.median(dist_matrix[np.triu_indices(K, k=1)]):.4f}')

fig = px.imshow(dist_matrix,
    x=[f'C{i}' for i in range(K)], y=[f'C{i}' for i in range(K)],
    color_continuous_scale='Viridis', aspect='equal',
    title='Cosine Distance Between Cluster Bio-Vectors',
    template=PLOTLY_DARK, width=650, height=550)
fig.show()

from sklearn.decomposition import PCA as PCA2
pca2d = PCA2(n_components=2)
bio_2d = pca2d.fit_transform(bio_normed.values)
print(f'\nPCA of bio-scores: {pca2d.explained_variance_ratio_} (cumulative: {pca2d.explained_variance_ratio_.sum():.3f})')

fig = go.Figure()
fig.add_trace(go.Scatter(x=bio_2d[:, 0], y=bio_2d[:, 1], mode='markers+text',
    text=[f'C{i}' for i in range(K)], textposition='top center',
    marker=dict(size=12, color=cluster_bio['health'].values, colorscale='RdYlGn', showscale=True,
                colorbar=dict(title='Health')),
    hovertext=[f'C{i}: health={cluster_bio.loc[i,"health"]:.2f}' for i in range(K)]))
fig.update_layout(title='Cluster Bio-Vectors in 2D (PCA of bio-scores)',
    xaxis_title='Bio-PC1', yaxis_title='Bio-PC2',
    template=PLOTLY_DARK, width=750, height=550)
fig.show()

Pairwise cosine distances between cluster bio-score vectors:
  Min (excl 0): 0.0000
  Max: 1.9986
  Mean: 0.9367
  Median: 0.6312



PCA of bio-scores: [0.76565414 0.15140879] (cumulative: 0.917)


In [6]:
corr = cluster_bio.corr()
fig = px.imshow(corr.values, x=list(corr.columns), y=list(corr.columns),
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1, aspect='equal',
    title='Correlation Between Bio-Score Dimensions (across 16 clusters)',
    template=PLOTLY_DARK, width=700, height=600)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

print('Highly correlated pairs (|r| > 0.9):')
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i, j]) > 0.9:
            print(f'  {corr.columns[i]} <-> {corr.columns[j]}: r={corr.iloc[i,j]:.3f}')

print('\nEffective dimensionality:')
from sklearn.decomposition import PCA as PCA3
pca_full = PCA3()
pca_full.fit(bio_normed.values)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
for i, cv in enumerate(cumvar):
    if i < 8:
        print(f'  PC{i+1}: {pca_full.explained_variance_ratio_[i]:.3f} (cumulative: {cv:.3f})')

Highly correlated pairs (|r| > 0.9):
  health <-> butyrate_SCFA: r=0.954
  health <-> tryptophan_serotonin: r=0.920
  health <-> dopamine_catecholamine: r=-0.962
  health <-> pro_inflammatory: r=-0.982
  health <-> fasting_inflammation: r=-0.997
  health <-> lipid_adverse: r=-0.998
  health <-> diet_hPDI: r=0.992
  health <-> diet_uPDI: r=-0.995
  butyrate_SCFA <-> dopamine_catecholamine: r=-0.953
  butyrate_SCFA <-> pro_inflammatory: r=-0.971
  butyrate_SCFA <-> fasting_inflammation: r=-0.961
  butyrate_SCFA <-> lipid_adverse: r=-0.938
  butyrate_SCFA <-> diet_hPDI: r=0.971
  butyrate_SCFA <-> diet_uPDI: r=-0.969
  tryptophan_serotonin <-> pro_inflammatory: r=-0.914
  tryptophan_serotonin <-> fasting_inflammation: r=-0.922
  tryptophan_serotonin <-> lipid_adverse: r=-0.927
  tryptophan_serotonin <-> diet_hPDI: r=0.915
  tryptophan_serotonin <-> diet_uPDI: r=-0.903
  dopamine_catecholamine <-> pro_inflammatory: r=0.972
  dopamine_catecholamine <-> fasting_inflammation: r=0.960
  dopami

---
## 3. Random Points in PCA Space → Bio-Scores

Sample random 3D PCA coordinates within the convex hull of the data, inverse-transform to 135D gut profiles, compute bio-scores, and compare to cluster centroids.

In [7]:
def make_diet(weights_dict):
    v = np.zeros(len(FOOD_GROUPS))
    for g, w in weights_dict.items():
        v[FOOD_GROUPS.index(g)] = w
    return v / v.sum()

ALL_ARCHETYPES = {}
for _, row in pd.read_csv(OUT_DIR / 'archetype_definitions.csv', index_col=0).iterrows():
    ALL_ARCHETYPES[row.name] = row.values

profiles, labels = [], []
arch_names = list(ALL_ARCHETYPES.keys())
arch_vecs = np.array(list(ALL_ARCHETYPES.values()))
for name, dv in ALL_ARCHETYPES.items():
    for _ in range(40):
        noisy = dv + np.random.normal(0, 0.02, len(dv))
        noisy = np.clip(noisy, 0, None); noisy /= noisy.sum()
        profiles.append(noisy @ corr_np)
        labels.append(name)
for _ in range(700):
    i1, i2 = np.random.choice(len(arch_names), 2, replace=False)
    t = np.random.beta(2, 2)
    bl = t * arch_vecs[i1] + (1-t) * arch_vecs[i2] + np.random.normal(0, 0.01, len(arch_vecs[0]))
    bl = np.clip(bl, 0, None); bl /= bl.sum()
    profiles.append(bl @ corr_np)
    labels.append('blend')

profiles = np.array(profiles)
scaler = StandardScaler()
profiles_scaled = scaler.fit_transform(profiles)
pca3 = PCA(n_components=3)
coords_3d = pca3.fit_transform(profiles_scaled)

print(f'PCA fit: {pca3.explained_variance_ratio_}, cumulative: {np.cumsum(pca3.explained_variance_ratio_)}')
print(f'PC1 range: [{coords_3d[:,0].min():.1f}, {coords_3d[:,0].max():.1f}]')
print(f'PC2 range: [{coords_3d[:,1].min():.1f}, {coords_3d[:,1].max():.1f}]')
print(f'PC3 range: [{coords_3d[:,2].min():.1f}, {coords_3d[:,2].max():.1f}]')

PCA fit: [0.57691477 0.15712186 0.06739789], cumulative: [0.57691477 0.73403664 0.80143453]
PC1 range: [-20.5, 16.9]
PC2 range: [-18.5, 9.4]
PC3 range: [-6.3, 13.3]


In [8]:
np.random.seed(99)
n_random = 500

pc1_range = [coords_3d[:,0].min(), coords_3d[:,0].max()]
pc2_range = [coords_3d[:,1].min(), coords_3d[:,1].max()]
pc3_range = [coords_3d[:,2].min(), coords_3d[:,2].max()]

random_3d = np.column_stack([
    np.random.uniform(pc1_range[0]*0.8, pc1_range[1]*0.8, n_random),
    np.random.uniform(pc2_range[0]*0.8, pc2_range[1]*0.8, n_random),
    np.random.uniform(pc3_range[0]*0.8, pc3_range[1]*0.8, n_random),
])

random_scaled = pca3.inverse_transform(random_3d)
random_original = scaler.inverse_transform(random_scaled)

random_bio = []
for i in range(n_random):
    gut = pd.Series(random_original[i], index=important_species)
    scores = compute_bio_scores(gut)
    random_bio.append(scores)
random_bio_df = pd.DataFrame(random_bio)

print(f'Random bio-scores: {random_bio_df.shape}')
print(f'\nRange per dimension:')
for col in random_bio_df.columns:
    rmin, rmax = random_bio_df[col].min(), random_bio_df[col].max()
    cmin, cmax = cluster_bio[col].min(), cluster_bio[col].max()
    print(f'  {col:25s}: random=[{rmin:+.4f}, {rmax:+.4f}]  clusters=[{cmin:+.4f}, {cmax:+.4f}]  cluster covers {(cmax-cmin)/(rmax-rmin)*100:.0f}% of random range')

Random bio-scores: (500, 12)

Range per dimension:
  health                   : random=[-1.4755, +3.8060]  clusters=[-1.3921, +3.5754]  cluster covers 94% of random range
  butyrate_SCFA            : random=[-0.0249, +0.0623]  clusters=[-0.0247, +0.0627]  cluster covers 100% of random range
  GABA                     : random=[-0.0318, +0.0135]  clusters=[-0.0230, +0.0126]  cluster covers 79% of random range
  tryptophan_serotonin     : random=[-0.0210, +0.0513]  clusters=[-0.0229, +0.0372]  cluster covers 83% of random range
  dopamine_catecholamine   : random=[-0.0955, +0.0207]  clusters=[-0.1066, +0.0181]  cluster covers 107% of random range
  pro_inflammatory         : random=[-0.0871, +0.0304]  clusters=[-0.0807, +0.0244]  cluster covers 89% of random range
  gut_barrier              : random=[-0.0177, +0.0135]  clusters=[-0.0102, +0.0193]  cluster covers 95% of random range
  fasting_inflammation     : random=[-0.5186, +0.1751]  clusters=[-0.4578, +0.1514]  cluster covers 88% of 

In [9]:
fig = make_subplots(rows=3, cols=4, subplot_titles=list(random_bio_df.columns))
for i, col in enumerate(random_bio_df.columns):
    row, c = i // 4 + 1, i % 4 + 1
    fig.add_trace(go.Histogram(x=random_bio_df[col], nbinsx=40, name='Random', opacity=0.6,
        marker_color='steelblue', showlegend=(i==0)), row=row, col=c)
    for ci in range(K):
        fig.add_trace(go.Scatter(x=[cluster_bio.loc[ci, col]], y=[0], mode='markers',
            marker=dict(size=8, color='red', symbol='diamond'),
            showlegend=(i==0 and ci==0), name='Cluster centroids'), row=row, col=c)

fig.update_layout(title='Bio-Score Distributions: 500 Random Points vs 16 Cluster Centroids',
    template=PLOTLY_DARK, height=700, width=1100, showlegend=True)
fig.show()

---
## 4. EMA Simulation → Bio-Score Trajectory

Simulate 500 food selections (random uniform from the food catalog) and track how the 12D bio-score vector evolves.

In [10]:
FOOD_CATEGORY_MAP = {}
for food_id in weight_matrix.index:
    row = weight_matrix.loc[food_id]
    groups = {g: w for g, w in row.items() if w > 0}
    if groups:
        FOOD_CATEGORY_MAP[food_id] = groups

all_food_ids = list(FOOD_CATEGORY_MAP.keys())
np.random.seed(123)
food_sequence = np.random.choice(all_food_ids, 500)

alpha = 0.08
gut = np.zeros(len(important_species))
trajectory_bio = []
diet_history = []

trajectory_bio.append(compute_bio_scores(gut))

for food_id in food_sequence:
    if food_id not in food_profiles_df.index:
        continue
    signal = food_profiles_df.loc[food_id].values
    gut = (1 - alpha) * gut + alpha * signal
    for fg, w in FOOD_CATEGORY_MAP.get(food_id, {}).items():
        diet_history.append(fg)
    trajectory_bio.append(compute_bio_scores(gut, diet_history[-200:]))

traj_df = pd.DataFrame(trajectory_bio)
print(f'Trajectory: {len(traj_df)} steps, {len(traj_df.columns)} bio-scores')
print(f'\nFinal bio-scores:')
for col in traj_df.columns:
    print(f'  {col:25s}: start={traj_df.iloc[0][col]:.4f} \u2192 end={traj_df.iloc[-1][col]:.4f}')

Trajectory: 501 steps, 12 bio-scores

Final bio-scores:
  health                   : start=0.0000 → end=1.7818
  butyrate_SCFA            : start=0.0000 → end=0.0266
  GABA                     : start=0.0000 → end=0.0036
  tryptophan_serotonin     : start=0.0000 → end=0.0247
  dopamine_catecholamine   : start=0.0000 → end=-0.0518
  pro_inflammatory         : start=0.0000 → end=-0.0372
  gut_barrier              : start=0.0000 → end=0.0051
  fasting_inflammation     : start=0.0000 → end=-0.2378
  lipid_adverse            : start=0.0000 → end=-0.1479
  diet_hPDI                : start=0.0000 → end=0.2422
  diet_uPDI                : start=0.0000 → end=-0.2451
  diversity_proxy          : start=0.0000 → end=0.8981


In [11]:
fig = make_subplots(rows=3, cols=4, subplot_titles=list(traj_df.columns))
for i, col in enumerate(traj_df.columns):
    row, c = i // 4 + 1, i % 4 + 1
    fig.add_trace(go.Scatter(y=traj_df[col], mode='lines', name=col,
        line=dict(width=1), showlegend=False), row=row, col=c)
    cmin, cmax = cluster_bio[col].min(), cluster_bio[col].max()
    fig.add_hrect(y0=cmin, y1=cmax, fillcolor='white', opacity=0.08,
        line_width=0, row=row, col=c)

fig.update_layout(title='Bio-Score Trajectory (500 random food selections, alpha=0.08)<br><sup>White bands = range covered by 16 cluster centroids</sup>',
    template=PLOTLY_DARK, height=700, width=1100)
fig.show()

In [12]:
traj_normed = traj_df.copy()
for col in traj_normed.columns:
    cmin = cluster_bio[col].min()
    cmax = cluster_bio[col].max()
    rng = cmax - cmin
    if rng > 0:
        traj_normed[col] = (traj_normed[col] - cmin) / rng
    else:
        traj_normed[col] = 0.5

print('Fraction of trajectory within cluster range [0, 1] per dimension:')
for col in traj_normed.columns:
    in_range = ((traj_normed[col] >= 0) & (traj_normed[col] <= 1)).mean()
    print(f'  {col:25s}: {in_range*100:.1f}% within cluster range')

print(f'\nOverall: {(traj_normed.ge(0).all(axis=1) & traj_normed.le(1).all(axis=1)).mean()*100:.1f}% of steps fully within cluster hull')

Fraction of trajectory within cluster range [0, 1] per dimension:
  health                   : 100.0% within cluster range
  butyrate_SCFA            : 100.0% within cluster range
  GABA                     : 99.6% within cluster range
  tryptophan_serotonin     : 100.0% within cluster range
  dopamine_catecholamine   : 100.0% within cluster range
  pro_inflammatory         : 100.0% within cluster range
  gut_barrier              : 100.0% within cluster range
  fasting_inflammation     : 100.0% within cluster range
  lipid_adverse            : 100.0% within cluster range
  diet_hPDI                : 100.0% within cluster range
  diet_uPDI                : 100.0% within cluster range
  diversity_proxy          : 0.0% within cluster range

Overall: 0.0% of steps fully within cluster hull


---
## 5. Nearest-Cluster Analysis

For each point along the trajectory, find the nearest cluster centroid in bio-score space. How stable is the assignment? How close is the nearest cluster?

In [13]:
cluster_bio_arr = cluster_bio.values

nearest_cluster = []
nearest_dist = []
second_dist = []

for i in range(len(traj_df)):
    point = traj_df.iloc[i].values
    dists = np.array([np.linalg.norm(point - cluster_bio_arr[ci]) for ci in range(K)])
    sorted_idx = np.argsort(dists)
    nearest_cluster.append(sorted_idx[0])
    nearest_dist.append(dists[sorted_idx[0]])
    second_dist.append(dists[sorted_idx[1]])

nearest_cluster = np.array(nearest_cluster)
nearest_dist = np.array(nearest_dist)
second_dist = np.array(second_dist)

fig = make_subplots(rows=3, cols=1, subplot_titles=[
    'Nearest Cluster ID Over Time',
    'Distance to Nearest Cluster',
    'Distance Ratio (nearest / 2nd nearest) \u2014 lower = more ambiguous'
], vertical_spacing=0.08)

fig.add_trace(go.Scatter(y=nearest_cluster, mode='markers', marker=dict(size=2, color=nearest_cluster, colorscale='Spectral'),
    showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(y=nearest_dist, mode='lines', line=dict(width=1, color='steelblue'),
    showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(y=nearest_dist/second_dist, mode='lines', line=dict(width=1, color='orange'),
    showlegend=False), row=3, col=1)
fig.add_hline(y=0.8, line_dash='dash', line_color='red', opacity=0.5, row=3, col=1)

fig.update_layout(template=PLOTLY_DARK, height=700, width=1000,
    title='Trajectory Relationship to Cluster Centroids in Bio-Score Space')
fig.show()

ambiguous = (nearest_dist / second_dist > 0.8).mean()
print(f'Ambiguous assignments (ratio > 0.8): {ambiguous*100:.1f}%')
print(f'Cluster visit counts: {dict(zip(*np.unique(nearest_cluster, return_counts=True)))}')

Ambiguous assignments (ratio > 0.8): 99.6%
Cluster visit counts: {np.int64(0): np.int64(221), np.int64(4): np.int64(3), np.int64(5): np.int64(5), np.int64(6): np.int64(73), np.int64(7): np.int64(5), np.int64(8): np.int64(27), np.int64(9): np.int64(122), np.int64(10): np.int64(45)}


---
## 6. Summary Assessment

In [14]:
print('='*70)
print('BIO-SCORE VIABILITY ASSESSMENT')
print('='*70)

print('\n1. CLUSTER DISTINCTNESS')
dists = squareform(pdist(cluster_bio.values, metric='euclidean'))
print(f'   Euclidean distance: min={dists[dists>0].min():.4f}, mean={dists[np.triu_indices(K,k=1)].mean():.4f}, max={dists.max():.4f}')
cos_dists = squareform(pdist(cluster_bio.values, metric='cosine'))
print(f'   Cosine distance:    min={cos_dists[cos_dists>0].min():.4f}, mean={cos_dists[np.triu_indices(K,k=1)].mean():.4f}, max={cos_dists.max():.4f}')

print('\n2. DIMENSION CORRELATIONS')
high_corr = []
corr = cluster_bio.corr()
for i in range(len(corr)):
    for j in range(i+1, len(corr)):
        if abs(corr.iloc[i,j]) > 0.85:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i,j]))
if high_corr:
    print(f'   {len(high_corr)} pairs with |r|>0.85 \u2014 consider merging or dropping redundant dimensions:')
    for a, b, r in high_corr:
        print(f'     {a} <-> {b}: r={r:.3f}')
else:
    print('   No pairs with |r|>0.85 \u2014 all dimensions contribute unique information')

print('\n3. RANDOM WALK COVERAGE')
for col in traj_normed.columns:
    in_range = ((traj_normed[col] >= 0) & (traj_normed[col] <= 1)).mean()
    status = 'OK' if in_range > 0.7 else 'WARN: trajectory often outside cluster range'
    print(f'   {col:25s}: {in_range*100:5.1f}% in cluster range  [{status}]')

print('\n4. ASSIGNMENT STABILITY')
changes = np.sum(nearest_cluster[1:] != nearest_cluster[:-1])
print(f'   Cluster changes: {changes}/{len(nearest_cluster)-1} steps ({changes/(len(nearest_cluster)-1)*100:.1f}%)')
print(f'   Ambiguous assignments (ratio>0.8): {ambiguous*100:.1f}%')

print('\n5. VERDICT')
print('   The 12D bio-score vector is viable for regression-based mapping if:')
print('   - Cluster centroids span the reachable bio-score space')
print('   - The random walk stays mostly within the cluster convex hull')
print('   - Dimensions are sufficiently decorrelated for meaningful interpolation')

BIO-SCORE VIABILITY ASSESSMENT

1. CLUSTER DISTINCTNESS
   Euclidean distance: min=0.0694, mean=1.9112, max=5.1566
   Cosine distance:    min=0.0000, mean=0.9367, max=1.9986

2. DIMENSION CORRELATIONS
   35 pairs with |r|>0.85 — consider merging or dropping redundant dimensions:
     health <-> butyrate_SCFA: r=0.954
     health <-> tryptophan_serotonin: r=0.920
     health <-> dopamine_catecholamine: r=-0.962
     health <-> pro_inflammatory: r=-0.982
     health <-> fasting_inflammation: r=-0.997
     health <-> lipid_adverse: r=-0.998
     health <-> diet_hPDI: r=0.992
     health <-> diet_uPDI: r=-0.995
     butyrate_SCFA <-> dopamine_catecholamine: r=-0.953
     butyrate_SCFA <-> pro_inflammatory: r=-0.971
     butyrate_SCFA <-> fasting_inflammation: r=-0.961
     butyrate_SCFA <-> lipid_adverse: r=-0.938
     butyrate_SCFA <-> diet_hPDI: r=0.971
     butyrate_SCFA <-> diet_uPDI: r=-0.969
     tryptophan_serotonin <-> dopamine_catecholamine: r=-0.891
     tryptophan_serotonin <-> 